<center>
    <img src="https://raw.githubusercontent.com/dvgodoy/PyTorch101_ODSC_Europe2020/master/images/linear_dogs.jpg" width="900">
</center>

# Yet another math for ML course

# Домашнее задание 1 (advanced): три системы координат одной поездки

>
> 17 августа 2018 года. 08:47. Сан-Франциско.

> Он выбежал из дома, прыгнул в тачку, бросил телефон в подстаканник и вдавил газ в пол. Этот ублюдок не заметил, что на его телефоне стоял логгер. Мы записали абсолютно всё. Ускорение, угловую скорость, показатели магнитометра, GPS... Мы даже залезли в CAN-шину его тачки и вытащили скорость колес и угол руля. Не то что бы хоть что-то из этого поможет его найти, но когда ты начинаешь собирать данные - остановиться практически невозможно.
>




Перед вами кусочек датасета [comma2k19](https://github.com/commaai/comma2k19) — 33 часа поездок по трассе 280, выложенных в открытый доступ. Для этой домашки мы взяли из него три минуты: перекрёсток с остановкой и поворотом, кусочек трассы и города. Телефон водителя висел на держалке. Мы сочли, что это скучно и повернули все показания датчиков так, как будто телефон валялся в подстаканнике под произвольным углом. Физика поездки от этого никак не изменилась, телефон жёстко связан с машиной, но система координат для сбора показателей стала другой. Базис поменялся.

Каждую секунду к нам приходит вектор из трёх координат $x, y, z$. Это ускорения машины в разных направлениях. В этой домашке вам предстоит поисследовать их и изучить движение машины в трёх системах координат:

- **телефон** — оси $x, y, z$ датчиков. В них приходят сырые числа, и в них ничего не понятно;
- **машина** — вперёд, вправо, вниз. В них понятно, где газ, где тормоз и где поворот;
- **мир** — север, восток, вниз. В них понятно, куда мы вообще едем.

Матрица — это переводчик между системами координат. Умножение на неё — это линейное преобразование, которое берёт вектор, записанный в одном базисе, и выдаёт тот же вектор в другом. Наша задача - вычислить по данным датчиков матрицы перехода. От телефона к машине и от машины к миру, а затем попытаться восстановить траекторию машины.

Заодно мы попытаемся на пальцах разобраться в зоопарке датчиков и поймём, какая мощь лежит у нас в карманах.

## Какие датчики бывают

**Автомобиль —** это компьютер на колёсах с десятками электронных блоков управления (ECU). Каждый блок отвечает за свою задачу: один управляет двигателем, другой — тормозами, третий — подушками безопасности. Все эти блоки постоянно собирают информацию с сотен датчиков — температуры, давления, скорости, положения педалей — и обмениваются ею между собой через специальную внутреннюю сеть, которая называется CAN-шина (Controller Area Network). Это как общий чат, где все устройства пишут друг другу сообщения.

Получить часть этих данных можно через стандартный разъём OBD-II, который обычно находится где-то под рулём. В него вставляется небольшой Bluetooth-адаптер, телефон подключается к нему и начинает читать автомобильную телеметрию. Набор доступных показателей ограничен, но для нас сейчас это не очень важно. Главное: машина умеет сама рассказать нам кое-что о своём движении.

Кроме машины у нас есть ещё и телефон. Это тоже маленький компьютер, только без колёс. Когда-то давно по нему можно было даже звонить, а теперь внутри него находится тик-ток и куча датчиков. Нас в первую очередь будут интересовать два: акселерометр и гироскоп.

**Акселерометр** измеряет ускорение телефона. Он отвечает на вопрос «В какую сторону меня сейчас толкают?».

Грубо говоря, акселерометр это крошечный грузик, подвешенный на микроскопических пружинках. Когда телефон начинает ускоряться, грузик смещается. Датчик измеряет это смещение и по нему понимает, в какую сторону и насколько сильно телефон ускоряется. Причём измерение происходит сразу по трём осям: $(a_x, a_y, a_z).$

Акселерометр чувствует не только движение машины, но и гравитацию. Поэтому если телефон просто спокойно лежит на столе, акселерометр всё равно по оси `z` показывает `g = 9.8 м/с²`.

Когда машина разгоняется, тормозит или поворачивает, датчик видит смесь из нескольких эффектов: движение автомобиля, гравитацию и ускорение при повороте. Более того, если телефон немного наклонить, часть гравитации внезапно появится на другой его оси. Для акселерометра это может выглядеть очень похоже на настоящее ускорение автомобиля. Датчик повернут относительно земли и движется вместе с машиной. Акселерометр не понимает, как меняется угол относительно земли и видит только ускорения.

**Гироскоп** в отличие от акселерометра разбирается в углах. Он отвечает на вопрос «Как быстро меня сейчас поворачивают?».

Гироскоп измеряет угловую скорость вокруг трех осей телефона $(\omega_x, \omega_y, \omega_z)$. Грубо говоря, гироскоп это микроскопическая вибрирующая масса. Когда телефон вращается, из-за эффекта Кориолиса эта масса начинает немного отклоняться в сторону. Чем быстрее вращается телефон, тем сильнее отклонение. Датчик измеряет его и получает угловую скорость.

Главное отличие от акселерометра в том, что масса закреплена на пружинках по-другому (после этой фразы у кого-то из физиков с потока подгорело).

Гироскоп практически не интересуется тем, разгоняется машина вперёд или стоит на месте. Он реагирует на вращение. Если машина едет прямо — гироскоп почти ничего не показывает. Если машина начинает поворачивать — появляется угловая скорость.

**Проблема заключается в том,** что всё это измеряется относительно собственных осей телефона. Он может валяться в машине кверху-ногами и данные будут приходить в такой ориентации. Поэтому прежде чем начать пользоваться потоком данных, нам нужно понять, как телефон лежит относительно машины. Тут и начинается задача из домашки. По гравитации можно легко понять, где низ. По движению машины и данным GPS можно понять, где находится вперед, а третью ось построить дополнением до ортогонального базиса.

Но и это ещё не всё. Нам нужно также понимать где север. Ни гироскоп, ни акселерометр за это не шарят. Тут появляется магнитометр, электронный компас, который помогает нам в этом разобраться и понять, как потенциально можно перейти из координат машины в координаты планеты. Его перед использованием обычно приходится калибровать, а также не забывать про магнитное склонение. Ведь северный магнитный полюс не совпадает с географическим и движется от Канады в сторону России.

Но про это вы знаете из 8 класса, с уроков Географии. Ведь в школе же всех учили прокладывать курс по компасу, да?

## Наши соглашения

Чтобы не утонуть в транспонированиях, заключим джентльменское соглашение:

1. Во всех формулах вектор - столбец. Поэтому переход между системами координат записывается как $a_{\text{машина}} = R \, a_{\text{телефон}}$.
2. В `NumPy` наблюдения лежат строками. Поэтому ровно тот же переход для всего массива записанный на питоне это `a_car = a_phone @ R.T`.
3. Оси машины: `(вперёд, вправо, вниз)`. Оси мира: `(север, восток, вниз)`.
4. Курс $\psi$ отсчитывается от севера по часовой стрелке, если смотреть сверху: север $0°$, восток $90°$.
5. Углы в градусах живут на окружности: разность курсов $370°$ и $10°$ равна нулю.

## Формат проверки работы: peer review

Каждая работа проверяется пятью вашими случайно выбранными однокурсниками. Вы также обязаны проверить пять чужих работ.

Проверяющий выставляет баллы по критериям из задания и оставляет содержательные комментарии: что сделано хорошо, где есть ошибки, что стоит исправить.

Итоговая оценка за задание складывается из трёх частей:

$$
\text{Оценка} = 0.8 \cdot \text{Решение} + 0.1 \cdot \text{Полезность ревью} + 0.1 \cdot \text{Согласованность ревью}
$$

- **80% решение** — это баллы за вашу собственную работу, полученные при проверке.
- **10% полезность ревью** — после проверки авторы работ оценивают, насколько полезными были полученные комментарии. Хорошее ревью должно быть конкретным, вежливым и помогать понять ошибки. Комментарии вида «всё ок», «неверно», «плохо» без объяснений считаются бесполезными.
- **10% согласованность ревью** — оценка проверяющего сравнивается с оценками других проверяющих. Если ваши баллы систематически сильно отличаются от общего консенсуса без убедительных комментариев, эта часть оценки снижается.

Максимум за домашку — 10 баллов, плюс 1.5 бонусных: 0.5 за задание 0 и 1 за задание 7\*. Итого можно набрать 11.5. Почти под каждым заданием есть вопросы. Ответы на них — часть задания и оцениваются наравне с кодом: работающий код с пустым «Ответ:» полного балла не получает.

Можно:

* указывать на ошибки и спорные места;
* снижать баллы, если решение не соответствует критериям;
* давать рекомендации по исправлению;
* писать развёрнутые комментарии по существу.

Нельзя:

* ставить баллы случайно или всем одинаково;
* писать формальные комментарии без содержания;
* оскорблять автора работы;
* договариваться о взаимных оценках;
* просить других поставить вам определённый балл;
* использовать peer review как способ отомстить или помочь друзьям.

Проверка тоже является частью задания. Халтурные, пустые или недобросовестные ревью будут снижать итоговый балл. Конкретные способы расчёта согласованности и полезности ревью не раскрываются.

Давайте попробуем сходу освоиться в системе и сделаем бонусное задание на бонусные баллы!

__(0.5 балла, бонус) Задание 0: знакомство__

**ФИО:**

**Напиши три факта о себе.** Один из них должен быть неправдой. Пусть проверяющие домашки угадают, какой:

1. ...
2. ...
3. ...


**Картинка, которая описывает вас лучше слов:**  вставьте её сюда! Мем, фото котика, любимый график — что угодно.

> **Для ревьюеров:** полный балл — если есть три факта и картинка. Скучные факты и скучная картинка баллы не снижают. Отсутствие картинки — снижает. В комментарии напишите, какой факт вы посчитали неправдой: угадали вы или нет — на оценку не влияет. Не забудьте поделиться упоротыми находками с чатиком.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import urllib.request

plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["font.size"] = 12

DATA_URL = "https://github.com/FUlyankin/yet_another_math_for_DS/raw/refs/heads/2026/data/trip_0817.npz"
DATA_FILE = Path("trip_0817.npz")

if not DATA_FILE.exists():
    urllib.request.urlretrieve(DATA_URL, DATA_FILE)

raw = np.load(DATA_FILE)

def load_segment(key):
    # возвращает словарь: имя датчика -> (t, value)
    names = sorted(set(k.split("/")[1] for k in raw.files if k.startswith(key + "/")))
    return {n: (raw[f"{key}/{n}/t"], raw[f"{key}/{n}/value"]) for n in names}

trip = {k: load_segment(k) for k in ["A", "B", "C"]}
for k, seg in trip.items():
    print(k, {n: v[1].shape for n, v in seg.items()})

Почти после каждого шага в этой домашке стоит ячейка с проверкой. Она не знает правильного ответа, но знает, каким он обязан быть: единичной длины, ортогональным, в разумном диапазоне. Если проверка прошла — можно идти дальше, если нет — сообщение подскажет, что именно не так. Запускайте её сразу, не копите ошибки.

In [ ]:
def check(condition, message):
    # маленький помощник для самопроверки: ✅ если всё хорошо, ошибка с объяснением — если нет
    if not condition:
        raise AssertionError("❌ " + message)
    print("✅", message)

# Часть 0: смотрим на данные

С датчиков записано три сегмента по 60 секунд:

* **A:** подъезд к перекрёстку, остановка, разгон и поворот направо, ещё одна остановка;
* **B:** трасса, 100–120 км/ч;
* **C:** снова город.

Внутри каждого лежат датчики, у каждого свой массив времени `t` (в секундах от начала сегмента) и массив значений. Из-за разной частоты нам придется постоянно сопоставлять разные оси времени друг с другом. Мы будем это делать функцией `np.interp`. Она строит линейную интерполяцию между двумя массивами.

In [ ]:
t_can = np.array([0.0, 1.0, 2.0])       # когда записан датчик
v_can = np.array([10.0, 12.0, 20.0])    # значение датчика

# мы не знаем значение датчика в момент 0.5 и 1.25, но можем интерполировать
# его как взвешенное среднее соседних значений
np.interp([0.5, 1.25], t_can, v_can)

| датчик | частота | что лежит в `value` | система координат |
|---|---|---|---|
| `acc` | 100 Гц | ускорение, м/с², 3 оси | телефон |
| `gyro` | 100 Гц | угловая скорость, рад/с, 3 оси | телефон |
| `gyro_uncalibrated` | 100 Гц | то же, но без вычета смещения нуля | телефон |
| `mag` | 10 Гц | магнитное поле, мкТл, 3 оси, сырое | телефон |
| `can_speed` | ~80 Гц | скорость машины, м/с | машина |
| `can_steering` | ~80 Гц | угол руля, градусы | машина |
| `can_wheel_speed` | ~80 Гц | скорости четырёх колёс, м/с | машина |
| `gps` | 5–10 Гц | широта, долгота, высота, скорость (м/с), курс (градусы от севера по часовой) | спутник |


Акселерометр, гироскоп и магнитометр записаны в системе координат телефона. Оси у неё правые: если $x$ и $y$ — это «вперёд» и «вправо», то $z$ — «вниз».


Скорость с CAN-шины и GPS — это независимые источники правды из других систем координат. Мы будем сверяться с ними.

In [ ]:
def plot_xyz(t, v, title, ylabel="", ax=None):
    ax = ax or plt.gca()
    for i, name in enumerate("xyz"):
        ax.plot(t, v[:, i], label=name, lw=0.8)
    ax.set_title(title); ax.set_xlabel("t, с"); ax.set_ylabel(ylabel); ax.legend(loc="upper right")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, key in zip(axes, "ABC"):
    t, a = trip[key]["acc"]
    plot_xyz(t, a, f"Сегмент {key}: акселерометр", "м/с²", ax)
plt.tight_layout(); plt.show()

t, a = trip["A"]["acc"]
t_v, v = trip["A"]["can_speed"]
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].plot(t, np.linalg.norm(a, axis=1), lw=0.8); ax[0].axhline(9.81, color="k", ls="--", lw=0.8)
ax[0].set_title("Сегмент A: длина вектора ускорения"); ax[0].set_ylabel("м/с²"); ax[0].set_xlabel("t, с")
ax[1].plot(t_v, v); ax[1].set_title("Сегмент A: скорость с CAN-шины"); ax[1].set_ylabel("м/с"); ax[1].set_xlabel("t, с")
plt.tight_layout(); plt.show()

Посмотрите на графики. Машина тормозит до полной остановки на десятой секунде, трогается, поворачивает, на 35-й секунде снова стоит. А длина вектора ускорения всё это время болтается около $9.81$.

**Разминка (не оценивается).**

Прежде чем читать дальше, запишите свою догадку одной строкой и не правьте её после решения.

Три оси акселерометра показывают примерно $(3.7,\ 7.2,\ -5.2)$ м/с² даже когда машина стоит (посмотрите на 10-ю секунду). Как по этим трём числам понять, где «низ», а где «вперёд»?  Можно ли это вообще понять по стоящей машине?

**Моя догадка:** ...

# Часть 1: Из телефона в машину

Нам нужно научиться переводить любой вектор из осей телефона в оси машины. Для этого достаточно знать три вещи: куда в координатах телефона смотрит «вперёд» машины, куда «вправо» и куда «вниз». Назовём эти три вектора $f$, $r$, $d$. Они единичной длины и перпендикулярны друг другу — как и положено осям.

Как ими пользоваться. Есть вектор $a$ в осях телефона. Сколько в нём «вперёд»? Это его проекция на $f$, то есть скалярное произведение $f^T a$. Сколько «вправо» - $r^T a$, сколько «вниз» - $d^T a$. Три скалярных произведения можно записать одним умножением на матрицу, если положить $f, r, d$ в её строки:

$$
R = \begin{pmatrix} f^T \\ r^T \\ d^T \end{pmatrix}, \qquad a_{\text{машина}} = R\, a_{\text{телефон}}.
$$

У такой матрицы есть два приятных свойства. Во-первых, обратный переход — это просто транспонирование: $R^{-1} = R^T$, никаких обратных матриц считать не надо. Во-вторых, $\det R = +1$ — это гарантия, что $R$ матрица поворота, а не отражения.

Осталось найти $f$, $r$, $d$ по данным. Никто нам их не скажет, но есть две подсказки от физики:

1. когда машина не ускоряется, акселерометр меряет только реакцию опоры — вектор длины $g$, направленный **вверх**. Отсюда узнаем $d$;
2. когда машина разгоняется или тормозит, к нему добавляется вектор вдоль оси **вперёд**, и его величина известна — это производная скорости с CAN-шины. Отсюда узнаем $f$;
3. вектор $r$ достроим дополнением до ортогонального базиса.

__(1 балл) Задание 1: ищем ось «вниз»__

Код — 0.5, вопросы — 0.5.

Возьмите сегмент A. Усредните показания акселерометра по всему сегменту и нормируйте результат на единичную длину. Это направление «вверх» в координатах телефона; «вниз» — противоположное.

Допишите функцию `estimate_down(acc)` и примените её к сегменту A. Выведите вектор $d$ и его длину.

Затем проделайте то же самое на сегментах B и C. Посчитайте угол между $d_A$ и $d_B$, $d_A$ и $d_C$ в градусах.

In [ ]:
def estimate_down(acc):
    # acc: массив (n, 3) в системе телефона
    # верните единичный вектор "вниз" в системе телефона

    # YOUR CODE HERE
    # ʕ•ᴥ•ʔ

    pass

def angle_between(u, v):
    # угол между векторами в градусах

    # YOUR CODE HERE
    # (⌐■_■)

    pass

d_A = estimate_down(trip["A"]["acc"][1])
print("вниз (телефон):", d_A.round(4), " длина:", np.linalg.norm(d_A).round(4))

In [ ]:
check(d_A.shape == (3,), "d — вектор из трёх чисел")
check(abs(np.linalg.norm(d_A) - 1) < 1e-10, "d единичной длины")
check(trip["A"]["acc"][1].mean(axis=0) @ d_A < 0, "d смотрит против среднего показания акселерометра: датчик меряет реакцию опоры, она направлена вверх, а d — вниз")
check(abs(angle_between(np.array([1, 0, 0]), np.array([0, 1, 0])) - 90) < 1e-9, "угол между ортами x и y — 90°")
check(abs(angle_between(d_A, d_A)) < 1e-6, "угол вектора с самим собой — 0° (следите за округлением в arccos)")
check(abs(angle_between(d_A, -d_A) - 180) < 1e-6, "угол с противоположным вектором — 180°")
check(all(angle_between(d_A, estimate_down(trip[k]["acc"][1])) < 5 for k in "BC"), "на B и C «вниз» смотрит туда же с точностью до пары градусов")

**Вопросы (0.5):**

1. Почему среднее по сегменту даёт направление гравитации, хотя машина всё время разгоняется и тормозит?
2. Почему $d$ на сегментах B и C отличается от $d_A$ на пару градусов, хотя телефон никто не трогал?
3. Для какого из трёх сегментов получается корректный результат? Почему результаты могут быть разными?

**Ответ:**

__(2.5 балла) Задание 2: ищем ось «вперёд»__

Акселерометр показывает сумму двух вещей: гравитацию и настоящее ускорение машины. Гравитация нам мешает, и мы уже знаем, как её убрать: она равна среднему показанию за сегмент. Вычитаем:

$$
a_{\text{lin}}(t) = a(t) - \bar{a}.
$$

Осталось только то, что делает машина. А делает она всего две вещи: разгоняется-тормозит и поворачивает.

* **Разгон и торможение** толкают вдоль оси «вперёд». Насколько сильно знает CAN-шина, с неё можно считать настоящее изменение скорости $\dot v(t)$ (эта запись означает производную скорости по времени, то есть ускорение).

* **Поворот** толкает вбок, вдоль оси «вправо», — это то самое чувство, когда на повороте прижимает к двери. Прижимает сильнее, если едем быстрее ($v$ больше) и если руль выкручен круче (машина быстрее поворачивается, угол $\omega_z$ больше). Тихо катиться на парковке с выкрученным рулём — не прижимает; лететь по трассе по еле заметной дуге — тоже почти не прижимает; а вот быстро войти в крутой поворот — прижимает как следует. Поэтому сила бокового толчка — это произведение $v \cdot \omega_z$.

Значит, в каждый момент времени ускорение машины — это «сколько вперёд» умножить на вектор «вперёд» плюс «сколько вбок» умножить на вектор «вправо»:

$$
a_{\text{lin}}(t) \approx \dot v(t) \cdot f + v(t) \cdot \omega_z(t) \cdot r.
$$

Векторы $f$ и $r$ неизвестны, всё остальное есть в данных. Перед нами обычная линейная регрессия, только таргетов три штуки сразу — по одному на каждую ось телефона. Признаков два: $\dot v$ и $v \cdot \omega_z$. Для каждой оси телефона мы подбираем два коэффициента: сколько в неё попадает от разгона и сколько от поворота.

- матрица признаков $X$ размера $n \times 2$: столбцы $\dot v$ и $v \cdot \omega_z$;
- матрица таргетов $Y$ размера $n \times 3$: три столбца $a_{\text{lin}}$;
- матрица коэффициентов $W$ размера $2 \times 3$, для которой $XW$ как можно ближе к $Y$.

Первая строка $W$ — три коэффициента при $\dot v$, по одному на ось телефона. Но это ровно и есть координаты вектора $f$ в осях телефона. Вторая строка — координаты $r$.

Задание большое, поэтому идём маленькими шагами. Для всех шагов понадобится сглаживание, функция `smooth` считает скользящее среднее по `k` точкам.

In [ ]:
def smooth(x, k=50):
    # скользящее среднее по k отсчётам вдоль первой оси
    kernel = np.ones(k) / k
    if x.ndim == 1:
        return np.convolve(x, kernel, mode="same")
    return np.column_stack([np.convolve(x[:, i], kernel, mode="same") for i in range(x.shape[1])])

**Шаг 2.1 (0.5): Приводим всё к одной сетке времени**

- У акселерометра и у CAN-шины свои массивы `t`, и они не совпадают. Возьмите скорость с CAN-шины и с помощью `np.interp` пересчитайте её в те моменты времени, в которые записан акселерометр.
- Теперь у вас есть $v(t)$ на сетке акселерометра — продифференцируйте её `np.gradient(v, t)`, получится $\dot v(t)$: насколько машина разгоняется или тормозит в каждый момент.
- Производная шумная, поэтому сгладьте её функцией `smooth`.

Функция `speed_on_acc_grid(seg)` должна вернуть два массива той же длины, что и время акселерометра: $v$ и сглаженную $\dot v$.

In [ ]:
def speed_on_acc_grid(seg):
    t, acc = seg["acc"]
    t_v, v_can = seg["can_speed"]

    # YOUR CODE HERE
    # ┌(ಠ_ಠ)┘

    return v, dv

v_A, dv_A = speed_on_acc_grid(trip["A"])

In [ ]:
t_A = trip["A"]["acc"][0]
check(v_A.shape == t_A.shape and dv_A.shape == t_A.shape, "v и dv той же длины, что время акселерометра")
check(abs(v_A[0] - trip["A"]["can_speed"][1][0]) < 0.1, "в нулевой момент скорость совпадает с CAN-шиной")
check(abs(np.sum(dv_A * np.gradient(t_A)) - (v_A[-1] - v_A[0])) < 1.0, "интеграл dv/dt по сегменту равен изменению скорости (с точностью до сглаживания на краях)")
check(np.abs(dv_A).max() < 6, "ускорение не превышает 6 м/с² — для легковой машины это потолок")

**Шаг 2.2 (0.5): Достаём скорость поворота**

Гироскоп выдаёт три числа: как быстро телефон вращается вокруг каждой из своих трёх осей. Но нас интересует одно-единственное вращение — вокруг вертикали, потому что именно так машина поворачивает на дороге. Наклоны кузова при разгоне и покачивания на кочках — тоже вращения, но нам они не нужны.

Вектор вертикали мы нашли, это $d$ из задания 1. Вращение вокруг $d$ — это часть вектора гироскопа, которая смотрит вдоль $d$, то есть проекция вектора гироскопа на $d$, то есть скалярное произведение `gyro @ d`.

Получится одно число в каждый момент времени: положительное — машина поворачивает направо, отрицательное — налево, ноль — едет прямо.

Гироскоп записан на своей сетке времени поэтому снова надо сделать `np.interp` на сетку акселерометра. Функция `yaw_rate(seg, d)` должна вернуть $\omega_z$ на сетке акселерометра, в рад/с.

In [ ]:
def yaw_rate(seg, d):
    t, acc = seg["acc"]
    t_g, gyro = seg["gyro"]

    # YOUR CODE HERE
    # (╯°□°)╯︵ ┻━┻

    return omega_z

omega_A = yaw_rate(trip["A"], d_A)

In [ ]:
check(omega_A.shape == t_A.shape, "omega_z на сетке акселерометра")
check(0.05 < np.abs(omega_A).max() < 1.0, "пиковая скорость поворота в разумных пределах (несколько градусов в секунду, не сотни)")
turn = np.degrees(np.sum(omega_A * np.gradient(t_A)))
check(60 < turn < 150, f"суммарный поворот за сегмент A: {turn:.0f}° — похоже на один поворот на перекрёстке")
check(turn > 0, "знак положительный: по правилу правой руки вокруг оси «вниз» это поворот по часовой, то есть направо")

**Шаг 2.3 (0.5): Линейная регрессия**

- Собираем матрицу $X$ размера $n \times 2$: первый столбец — $\dot v$, второй — $v \cdot \omega_z$.
- Матрица $Y$ размера $n \times 3$ — это сглаженное $a_{\text{lin}} = a - \bar a$.
- Мы ищем $W$ размера $2 \times 3$, в этом поможет `np.linalg.lstsq(X, Y, rcond=None)`.

Первая строка $W$ — это направление, вдоль которого «живёт» разгон, то есть $f$, вторая — направление, вдоль которого живёт поворот, то есть $r$.

Функция `fit_axes(seg, d)` должна вернуть две строки $W$: `f_raw` и `r_raw`.

In [ ]:
def fit_axes(seg, d):
    t, acc = seg["acc"]
    v, dv = speed_on_acc_grid(seg)
    omega_z = yaw_rate(seg, d)

    # YOUR CODE HERE
    # (づ｡◕‿‿◕｡)づ

    return f_raw, r_raw

f_raw, r_raw = fit_axes(trip["A"], d_A)
print("f_raw =", f_raw.round(3), " r_raw =", r_raw.round(3))

In [ ]:
check(f_raw.shape == (3,) and r_raw.shape == (3,), "f_raw и r_raw — векторы из трёх чисел")
check(0.6 < np.linalg.norm(f_raw) < 1.3, f"длина f_raw ≈ {np.linalg.norm(f_raw):.2f}: около единицы, ведь dv/dt уже в м/с²")
check(abs(angle_between(f_raw, d_A) - 90) < 10, f"f_raw почти перпендикулярен d: угол {angle_between(f_raw, d_A):.1f}°")
check(angle_between(r_raw, np.cross(d_A, f_raw)) < 25, "r_raw смотрит примерно туда же, куда d × f: регрессия поймала физику")

**Шаг 2.4 (0.5): Приводим оси в порядок**

Регрессия подобрала $f$ и $r$ так, чтобы формула лучше всего описывала данные. Но у неё не было ни одного ограничения на то, какими эти векторы должны быть. А мы знаем, какими: оси машины — единичной длины и перпендикулярны друг другу. Регрессионные $f$ и $r$ этому не удовлетворяют.

- **Длина.** Она у $f$ примерно единица, но не ровно: акселерометр слегка врёт в масштабе, а CAN-шина — в скорости. Регрессия честно подстроила длину под данные, а нам нужна ровно единица, иначе матрица не будет поворотом.
  
- **Перпендикулярность к $d$.** Модель «ускорение = разгон + поворот» неполная: при торможении нос машины клюёт вниз, при разгоне задирается. В эти моменты кусочек гравитации попадает в горизонтальные оси, и регрессия, чтобы это объяснить, чуть наклоняет $f$ к вертикали. Проверка в 2.3 как раз это показала: угол между $f$ и $d$ не ровно $90°$, а $91$–$92°$.

- **Вектор $r$ шумный.** Боковых ускорений на сегменте мало и они слабые, так что второй столбец регрессии оценён гораздо хуже первого. Чинить его бессмысленно — проще собрать заново.

Из трёх векторов доверяем в первую очередь $d$. Гравитация бессердечная сволочь, но она очень надежная.

Займёмся починкой результатов:

1. $f$: вычитаем из него проекцию на $d$, то есть $f = f - \langle f, d \rangle \cdot d$ — теперь он строго перпендикулярен $d$, — и делим на длину. Это один шаг ортогонализации Грама–Шмидта.
2. $r$: из регрессии выбрасываем и берём `np.cross(d, f)`: векторное произведение двух единичных перпендикулярных векторов — это единичный вектор, перпендикулярный обоим.
3. Кладём $f, r, d$ построчно — это и есть матрица поворота $R$.

Функция `build_rotation(f_raw, d)` должна вернуть матрицу $3 \times 3$.

In [ ]:
def build_rotation(f_raw, d):

    # YOUR CODE HERE
    # (ﾉ◕ヮ◕)ﾉ*:･ﾟ✧

    return R

R_A = build_rotation(f_raw, d_A)
print(np.round(R_A, 4))

In [ ]:
check(R_A.shape == (3, 3), "R — матрица 3×3")
check(np.abs(R_A @ R_A.T - np.eye(3)).max() < 1e-10, "R Rᵀ = I: строки ортонормированы")
check(abs(np.linalg.det(R_A) - 1) < 1e-10, "det R = +1: это поворот, а не отражение")
check(np.allclose(R_A[2], d_A), "третья строка R — это d")
check(abs(R_A[0] @ d_A) < 1e-10, "первая строка перпендикулярна d")

**Вопросы (0.5):**

Что будет, если оставить в $X$ только один столбец $\dot v$? Почему на сегменте A это ещё работает, а если запустить `build_rotation` на сегменте B — нет?

**Hint:** Посмотрите на график скорости на B.

**Ответ:**

__(1 балл) Задание 3: сверяемся с CAN-шиной__

У нас есть $R_A$, и мы утверждаем, что это матрица перехода из телефона в машину. Как проверить, что мы не ошиблись? Так же, как проверяют любую модель — на данных, которые в её построении не участвовали. У нас есть независимый источник: CAN-шина. Она ничего не знает про телефон, зато знает, как машина разгоняется (скорость колёс) и как поворачивает (угол руля).

Идея простая. Если $R_A$ правильная, то после перевода в систему машины ускорение по оси «вперёд» должно вести себя как $\dot v$ с CAN-шины, а вращение вокруг оси «вниз» — как угол руля. Если матрица кривая, оси перепутаются, и ничего похожего не получится.

**Шаг 3.1 (0.5): Переезжаем в машину**

Допишите `to_car_frame(seg, R)`, которая переводит оба датчика в систему машины:

- акселерометр: вычтите гравитацию (среднее по сегменту) и умножьте на `R.T`;
- гироскоп: сначала положите его на сетку времени акселерометра — `np.interp` для каждого из трёх столбцов, — потом умножьте на `R.T`.

Гироскоп переводится той же матрицей, что и акселерометр: $R_A$ описывает, как телефон лежит в машине, а не какой датчик мы читаем, и оба датчика записаны в одних и тех же осях телефона.

Функция должна вернуть `t`, `a_car` и `w_car` — оба массива формы `(n, 3)` на сетке акселерометра.

In [ ]:
def to_car_frame(seg, R):
    t, acc = seg["acc"]
    t_g, gyro = seg["gyro"]

    # YOUR CODE HERE
    # [̲̅$̲̅(̲̅5̲̅)̲̅$̲̅]

    return t, a_car, w_car

t, a_car, w_car = to_car_frame(trip["A"], R_A)

In [ ]:
check(a_car.shape == (len(t), 3) and w_car.shape == (len(t), 3), "a_car и w_car — массивы (n, 3) на сетке акселерометра")
check(np.abs(a_car.mean(axis=0)).max() < 1e-6, "среднее a_car — ноль: гравитацию вычли")
check(np.corrcoef(smooth(a_car[:, 0], 20), smooth(dv_A, 20))[0, 1] > 0.8, "a_вперёд повторяет dv/dt с CAN-шины (corr > 0.8)")
check(np.allclose(w_car[:, 2], omega_A), "третья компонента w_car — это ω_z из шага 2.2")

**Шаг 3.2: Смотрим на графики**

`compare_with_can` уже написана: она рисует три пары графиков и считает корреляцию в каждой паре. Запустите её для всех трёх сегментов и посмотрите на картинки:

- **$a_{\text{вперёд}}$ и $\dot v$ с CAN-шины.** Синяя линия — что чувствует телефон вдоль оси «вперёд», оранжевая — что говорят колёса. Если оси найдены верно, они ходят вместе: тормозим — обе вниз, разгоняемся — обе вверх.
- **$\omega_{\text{вниз}}$ и угол руля.** Скорость поворота по гироскопу против того, насколько выкручен руль. Должны быть зеркальны друг другу: где руль повёрнут, там и вращение.
- **$a_{\text{вправо}}$ и $v \cdot \omega_{\text{вниз}}$.** Боковое ускорение против того, что мы предсказали в задании 2. Тоже должны совпадать.

В заголовке каждого графика — корреляция. Близко к $\pm 1$ — оси найдены правильно, близко к нулю — что-то перепутано. Матрица одна на всю поездку: телефон никто не трогал, так что на B и C она должна работать не хуже, чем на A.


In [ ]:
def compare_with_can(seg, R, title):
    t, a_car, w_car = to_car_frame(seg, R)
    v, dv = speed_on_acc_grid(seg)
    t_s, steer = seg["can_steering"]
    steer_i = np.interp(t, t_s, steer)
    pairs = [
        ("a_вперёд", smooth(a_car[:, 0], 20), "dv/dt (CAN)", dv),
        ("ω_вниз", w_car[:, 2], "угол руля (CAN)", steer_i),
        ("a_вправо", smooth(a_car[:, 1], 20), "v·ω_вниз", smooth(v * w_car[:, 2], 20)),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(16, 3.5))
    out = []
    for ax, (n1, x1, n2, x2) in zip(axes, pairs):
        ax.plot(t, x1, label=n1, lw=0.9)
        ax2 = ax.twinx(); ax2.plot(t, x2, color="C1", label=n2, lw=0.9)
        c = np.corrcoef(x1, x2)[0, 1]; out.append(c)
        ax.set_title(f"{title}: {n1} vs {n2}, corr = {c:.2f}"); ax.set_xlabel("t, с")
    plt.tight_layout(); plt.show()
    return tuple(out)

corrs = {key: compare_with_can(trip[key], R_A, f"Сегмент {key}") for key in "ABC"}

**Вопросы (0.5):**

1. Почему мы перевели показания гироскопа в координаты машины матрицей, полученной по акселерометру?

2. На сегменте B корреляция $a_{\text{вперёд}}$ с $\dot v$ заметно ниже, чем на A. Почему? Что на трассе портит продольную ось, но не портит боковую?

3. Корреляция $\omega_{\text{вниз}}$ с углом руля получилась отрицательной. Это ошибка в $R$ или нет? В какую сторону машина поворачивает на сегменте A?

**Hint:** посмотрите на курс GPS, он растёт от 170° к 280°, какой знак у $\omega_{\text{вниз}}$ при таком повороте по правилу правой руки, и какой знак у угла руля?


**Ответ:**

# Часть 2: из машины в настоящий мир

Ура! Линейная регрессия, ортогонализация Грама–Шмидта и здравый смысл позволили нам переводить показания в оси машины. Но машина поворачивает, и её «вперёд» всё время меняется относительно мира. Чтобы нарисовать траекторию на карте — север вверху, восток справа, — нужно знать, куда машина смотрит в каждый момент.

Это описывается одним числом — **курсом** $\psi$: углом между севером и осью «вперёд», по часовой стрелке. Курс $0°$ — едем на север, $90°$ — на восток.

Мировые оси у нас: север, восток, вниз. «Вниз» общая с машиной (дорогу считаем плоской), так что переход из машины в мир — это поворот вокруг вертикали на угол $\psi$:

$$
R_z(\psi) = \begin{pmatrix} \cos\psi & -\sin\psi & 0 \\ \sin\psi & \cos\psi & 0 \\ 0 & 0 & 1 \end{pmatrix}, \qquad a_{\text{мир}} = R_z(\psi)\,a_{\text{машина}}.
$$

Например, ось «вперёд» $(1, 0, 0)$ переходит в $(\cos\psi, \sin\psi, 0)$. При $\psi = 90°$ это $(0, 1, 0)$ — восток.

Осталось узнать $\psi(t)$. Источников три, и у каждого свой характер:

* **GPS** сообщает курс напрямую, но редко, с задержкой и только в движении. В построении курса мы его использовать не будем — только для проверки;
* **гироскоп** сообщает, как быстро курс меняется. Если знать курс в начале и сложить все изменения, получим курс в любой момент;
* **магнитометр** показывает на север, всегда, без накопления ошибки. Но в машине полно железа, и оно искажает показания.

Начнём с гироскопа.

__(1 балл) Задание 4: курс из гироскопа__

**Шаг 4.1 (0.2): Разность углов**

Курс живёт на окружности: $370°$ и $10°$ - это одно и то же.

Напишите `angle_diff(a, b)`, которая возвращает разность $a - b$, приведённую к интервалу $(-180°, 180°]$. Она должна работать и с числами, и с массивами.

**Hint** нужна одна строчка с `% 360`

In [ ]:
def angle_diff(a, b):

    # YOUR CODE HERE
    # ʕ•ᴥ•ʔ
    pass

def around(a, ref):
    # угол a, записанный ближайшим к ref числом, чтобы графики не рвались на 0/360
    return ref + angle_diff(a, ref)

In [ ]:
check(angle_diff(370, 10) == 0, "370° и 10° — один и тот же курс")
check(angle_diff(0, 350) == 10, "от 350° до 0° — десять градусов вперёд, а не −350")
check(angle_diff(-170, 170) == 20, "−170° и 170° отличаются на 20°, через юг")
check(np.array_equal(angle_diff(np.array([0, 90, 180]), 45), np.array([-45, 45, 135])), "работает с массивами")

**Шаг 4.2 (0.3): Складываем повороты**

Гироскоп в системе машины даёт $\omega_{\text{вниз}}$, то есть на сколько градусов в секунду машина поворачивает прямо сейчас. За маленький шаг времени $dt$ она повернулась на $\omega_{\text{вниз}} \cdot dt$. Курс в момент $t$ — это сумма всех таких поворотов от начала сегмента плюс начальный курс. Его гироскоп не знает и мы добавим его чуть позже.

Напишите `gyro_heading(seg, R)`:

- переведите гироскоп в систему машины функцией `to_car_frame`;
- возьмите $\omega_{\text{вниз}}$, умножьте на шаг времени `np.gradient(t)` и накопите `np.cumsum`;
- переведите в градусы функцией `np.degrees`.

In [ ]:
def gyro_heading(seg, R):

    # YOUR CODE HERE
    # (⌐■_■)

    return t, psi

t_psi, psi_gyro = gyro_heading(trip["A"], R_A)

In [ ]:
check(psi_gyro.shape == t_psi.shape, "курс на сетке акселерометра")
check(abs(psi_gyro[0]) < 1, "в нулевой момент курс равен нулю — константу интегрирования ещё не знаем")
check(60 < psi_gyro[-1] - psi_gyro[0] < 150, f"за сегмент A курс изменился на {psi_gyro[-1] - psi_gyro[0]:.0f}° — как и суммарный поворот в шаге 2.2")

**Шаг 4.3 (0.25): Проверяем форму курса по GPS**

Гироскоп не знает начальный курс. Его придётся подобрать по GPS. Но подбирать будем только по **первым 5 секундам движения** (там, где машина едет быстрее 3 м/с, — на стоящей машине курс GPS не определён). Это калибровочный отрезок. Всё остальное будет тестовой выборкой.

Функция `compare_heading` ниже почти готова: маски `moving`, `calib` и `test` уже посчитаны, график уже нарисован. Допишите пять строк:

- `psi_i` — курс $\psi$, пересчитанный на моменты времени GPS (`np.interp`);
- `offset` — среднее `angle_diff(bearing, psi_i)` по калибровочным точкам;
- `psi_i` — тот же курс, сдвинутый на `offset`;
- `err` — `angle_diff(psi_i, bearing)` по тестовым точкам;
- `rmse` — корень из среднего квадрата `err`.

Запустите ячейку — она построит графики для сегментов A, B и C.

In [ ]:
def compare_heading(t, psi, seg, title, v_min=3.0, t_calib=5.0):
    t_gps, gps = seg["gps"]
    t_v, v_can = seg["can_speed"]
    bearing = gps[:, 4]

    moving = np.interp(t_gps, t_v, v_can) > v_min
    t_start = t_gps[moving][0]                       # первый момент движения
    calib = moving & (t_gps < t_start + t_calib)
    test = moving & ~calib

    # YOUR CODE HERE
    # ┌(ಠ_ಠ)┘

    psi_i = ...      # курс на моментах времени GPS
    offset = ...     # среднее расхождение с GPS по калибровочным точкам
    psi_i = ...      # сдвинутый курс
    err = ...        # расхождение с GPS по тестовым точкам
    rmse = ...

    ref = bearing[moving][0]
    plt.figure(figsize=(12, 3.5))
    plt.axvspan(t_start, t_start + t_calib, color="0.9", label="калибровка")
    plt.plot(t_gps[moving], around(bearing, ref)[moving], ".", ms=3, label="GPS")
    plt.plot(t_gps, around(psi_i, ref), lw=1, label="оценка")
    plt.title(f"{title}: RMSE на тесте = {rmse:.2f}°"); plt.ylabel("курс, °"); plt.xlabel("t, с"); plt.legend()
    plt.show()
    return psi + offset, rmse

results = {}
for key in "ABC":
    t, psi = gyro_heading(trip[key], R_A)
    results[key] = compare_heading(t, psi, trip[key], f"Сегмент {key}: курс из гироскопа")

In [ ]:
psi_A, rmse_A = results["A"]
t_gps, gps = trip["A"]["gps"]; t_v, v_can = trip["A"]["can_speed"]
moving = np.interp(t_gps, t_v, v_can) > 3
first5 = moving & (t_gps < t_gps[moving][0] + 5)

check(abs(angle_diff(np.interp(t_gps, t_psi, psi_A), gps[:, 4])[first5].mean()) < 0.5, "на калибровочном отрезке среднее расхождение с GPS — ноль")
check(rmse_A < 3, f"сегмент A: RMSE на тесте {rmse_A:.2f}° — гироскоп за минуту почти не уплыл")
check(all(results[k][1] < 3 for k in "BC"), "на B и C тоже меньше 3°")

**Вопрос (0.25):**

Подберите сдвиг по всему сегменту, а не по первым 5 секундам, и посчитайте RMSE ещё раз. Он станет меньше. Почему эта оценка качества — оптимистичная, и что общего у этой ситуации с оценкой модели на обучающей выборке?


**Ответ:**

Форму курса гироскоп восстанавливает отлично, RMSE около градуса. Но у него есть дыра: он не знает, откуда начать. Начальный курс мы взяли у GPS — а GPS мы договорились в построении не использовать, он только для проверки. Так что этот шаг был подглядыванием в ответ. Нужен независимый источник, который знает, где север, сам по себе.

__(2 балла) Задание 5: магнитометр__

Нам нужен датчик, который знает абсолютное направление без всяких спутников. Это магнитометр — обычный компас, только электронный. Он выдаёт вектор, который смотрит на север. С ним всё наоборот по сравнению с гироскопом: он не накапливает ошибку со временем, но каждое отдельное показание довольно шумное. В задании 6 мы их поженим: форму курса возьмём у гироскопа, а начальный курс — у магнитометра, и GPS в этой цепочке участвовать не будет. Но сначала надо научиться доставать курс из магнитометра — и починить его, потому что в машине он врёт.

Итак, магнитометр выдаёт вектор на север. Переведём его в систему машины и посмотрим на две горизонтальные компоненты: $m_{\text{вперёд}}$ — сколько севера впереди, $m_{\text{вправо}}$ — сколько севера справа.

Встаньте и покрутитесь на месте. Север никуда не девается, но относительно вас он уезжает: повернулись направо - север оказался слева. Значит, по тому, где относительно машины сидит север, видно, куда смотрит машина:

* север впереди - курс $0°$;
* север слева - машина смотрит на восток, курс $90°$;
* север сзади - курс $180°$;
* север справа - машина смотрит на запад, курс $270°$.

Курс — это просто угол между осью «вперёд» и вектором на север. Угол между двумя векторами мы считать умеем ещё с задания 1, через косинус. Одна беда: косинус не различает «повернулись на $90°$ вправо» и «на $90°$ влево» — в обоих случаях угол $90°$. Сторону подскажет знак $m_{\text{вправо}}$: если север справа, машина смотрит левее севера, и курс надо считать в другую сторону.

Но компас в машине врёт. Кузов, динамики, проводка - всё это намагничено и едет вместе с машиной. Магнитометр меряет сумму: поле Земли плюс поле машины. Поле Земли относительно машины крутится, когда машина поворачивает. А поле машины относительно машины не крутится никогда - машина сама относительно себя не поворачивается. Значит, это постоянная добавка $b$:

$$
m_{\text{измерено}} = m_{\text{земля}} + b.
$$

Как это выглядит на картинке: когда машина поворачивает, конец вектора Земли рисует окружность вокруг нуля. Измеренный вектор рисует ту же окружность, только сдвинутую в точку $b$. Так что план простой: нарисовать точки, найти центр окружности, вычесть его - и компас снова показывает на север.

**Шаг 5.1 (0.25): Смотрим на магнитометр в системе машины**

Переведите магнитометр сегмента A в систему машины (`@ R_A.T`). Нарисуйте точки $(m_{\text{вперёд}}, m_{\text{вправо}})$ на плоскости, покрасив их по времени. Должна получиться дуга окружности, и её центр — далеко не в нуле.

In [ ]:
t_m, mag = trip["A"]["mag"]

# YOUR CODE HERE
# (ﾉ◕ヮ◕)ﾉ*:･ﾟ✧

# m_car = ...

plt.figure(figsize=(5, 5))
plt.scatter(m_car[:, 0], m_car[:, 1], s=6, c=t_m, cmap="viridis")
plt.plot(0, 0, "k+", ms=12)
plt.axis("equal"); plt.xlabel("m вперёд, мкТл"); plt.ylabel("m вправо, мкТл"); plt.title("Сегмент A: магнитометр в системе машины"); plt.show()

In [ ]:
check(m_car.shape == mag.shape, "магнитометр повёрнут, размер не изменился")
check(np.allclose(np.linalg.norm(m_car, axis=1), np.linalg.norm(mag, axis=1)), "поворот не меняет длину векторов")
check(np.linalg.norm(m_car[:, :2].mean(axis=0)) > 50, "облако точек сидит далеко от нуля: поле Земли — около 50 мкТл, а тут явно что-то ещё")

**Шаг 5.2 (0.5): ищем центр окружности**

Окружность задаётся центром $(c_x, c_y)$ и радиусом $\rho$: $(x - c_x)^2 + (y - c_y)^2 = \rho^2$. Неизвестные тут сидят в квадратах, но если раскрыть скобки и перенести всё известное направо, получится

$$
2 c_x\, x + 2 c_y\, y + k = x^2 + y^2, \qquad k = \rho^2 - c_x^2 - c_y^2.
$$

Слева три неизвестных $(c_x, c_y, k)$ входят линейно, а $x$, $y$ и $x^2 + y^2$ мы знаем в каждой точке. Это снова линейная регрессия: матрица $n \times 3$ со столбцами $2x$, $2y$, $1$, правая часть $x^2 + y^2$, решаем `np.linalg.lstsq`. Из ответа достаём $c_x$, $c_y$ и $\rho = \sqrt{k + c_x^2 + c_y^2}$.

Напишите `fit_circle(x, y)`. Проверка ниже сначала прогонит её на игрушечной окружности с известным центром, и только потом — на магнитометре. Нарисуйте найденную окружность поверх точек.

In [ ]:
def fit_circle(x, y):

    # YOUR CODE HERE
    # (╯°□°)╯︵ ┻━┻

    return cx, cy, rho

cx, cy, rho = fit_circle(m_car[:, 0], m_car[:, 1])
print(f"центр окружности (hard iron): ({cx:.1f}, {cy:.1f}) мкТл, радиус {rho:.1f} мкТл")

In [ ]:
# сначала — игрушечная окружность: центр (3, −4), радиус 5, четверть дуги, чуть-чуть шума
rng = np.random.default_rng(0)
phi_test = np.linspace(0, np.pi / 2, 60)
x_test = 3 + 5 * np.cos(phi_test) + rng.normal(0, 0.02, 60)
y_test = -4 + 5 * np.sin(phi_test) + rng.normal(0, 0.02, 60)
cx_t, cy_t, rho_t = fit_circle(x_test, y_test)

check(abs(cx_t - 3) < 0.1 and abs(cy_t + 4) < 0.1, f"игрушечная окружность: центр найден ({cx_t:.2f}, {cy_t:.2f}) вместо (3, −4)")
check(abs(rho_t - 5) < 0.1, f"игрушечная окружность: радиус {rho_t:.2f} вместо 5")

# теперь — магнитометр
check(10 < rho < 60, f"радиус {rho:.0f} мкТл — порядок поля Земли, а не железа")
check(np.hypot(cx, cy) > 100, f"центр в {np.hypot(cx, cy):.0f} мкТл от нуля: железа в машине много")

**Шаг 5.3 (0.5): Курс из магнитометра**

Центр окружности найден — значит, железо машины мы знаем и можем его вычесть. После этого вектор магнитометра снова честно смотрит на север, и остаётся достать из него курс.

Вспомним, как это делается. Курс - это угол между осью «вперёд» и вектором на север. Угол между двумя векторами считается через косинус: $\cos\theta = \dfrac{\langle u, v \rangle}{|u|\,|v|}$. Здесь $u$ — ось «вперёд», то есть $(1, 0)$, а $v$ — горизонтальная часть магнитометра $(x, y)$. Скалярное произведение с $(1, 0)$ — это просто $x$, а длина $v$ — это $\sqrt{x^2 + y^2}$:

$$
\theta = \arccos\frac{x}{\sqrt{x^2 + y^2}}.
$$

Арккосинус даёт угол от $0°$ до $180°$ и не различает, в какую сторону машина отвернулась от севера. Сторону подсказывает $y$: если север справа ($y > 0$), машина смотрит левее севера, и курс равен $360° - \theta$. Если север слева или прямо впереди ($y \le 0$), курс равен $\theta$.

Допишите `mag_heading(seg, R, center)` — четыре строки:

- `x` и `y` — магнитометр в системе машины (`@ R.T`) минус центр окружности: `x = m_вперёд − c_x`, `y = m_вправо − c_y`;
- `theta` — угол по формуле выше, в градусах (`np.degrees`);
- `psi` — курс: `np.where(y > 0, 360 - theta, theta)`.

Функция `compare_mag` уже написана: она рисует курс из магнитометра рядом с курсом GPS и пишет в заголовок среднее расхождение и его разброс. Обратите внимание: сдвиг мы теперь не подбираем — магнитометр, в отличие от гироскопа, сам знает, где север.

Среднее расхождение с GPS получится не нулевым, а около $-8°$. Это магнитное склонение: компас показывает на магнитный полюс, а GPS считает курс от географического, и в Сан-Франциско они расходятся заметно. Найдите в интернете магнитное склонение для Сан-Франциско в 2018 году и сравните со своим числом. Дальше оно нам понадобится.

In [ ]:
def mag_heading(seg, R, center):
    t_m, mag = seg["mag"]

    # YOUR CODE HERE
    # (づ｡◕‿‿◕｡)づ

    x = ...
    y = ...
    theta = ...
    psi = ...        # север справа 360 - theta, иначе theta
    return t_m, psi


def compare_mag(seg, R, center, title, v_min=3.0):
    t_m, psi = mag_heading(seg, R, center)
    t_gps, gps = seg["gps"]; t_v, v_can = seg["can_speed"]
    bearing = np.interp(t_m, t_gps, gps[:, 4])
    moving = np.interp(t_m, t_v, v_can) > v_min
    err = angle_diff(psi, bearing)[moving]
    ref = bearing[moving][0]
    plt.figure(figsize=(12, 3.5))
    plt.plot(t_m[moving], around(bearing, ref)[moving], ".", ms=3, label="GPS"); plt.plot(t_m, around(psi, ref), lw=1, label="магнитометр")
    plt.title(f"{title}: среднее расхождение {err.mean():.1f}°, разброс {err.std():.1f}°")
    plt.ylabel("курс, °"); plt.xlabel("t, с"); plt.legend(); plt.show()
    return err

err_A = compare_mag(trip["A"], R_A, (cx, cy), "Сегмент A: курс из магнитометра")

In [ ]:
t_m_chk, psi_m_chk = mag_heading(trip["A"], R_A, (cx, cy))
check(np.all((psi_m_chk >= 0) & (psi_m_chk <= 360)), "курс из магнитометра лежит в [0, 360]")

# игрушечная проверка: север впереди, север слева, север справа
fake = {"mag": (np.array([0.0, 1.0, 2.0]), np.array([[1.0, 0.0, 0.0], [0.0, -1.0, 0.0], [0.0, 1.0, 0.0]]))}
_, psi_fake = mag_heading(fake, np.eye(3), (0.0, 0.0))

check(np.allclose(psi_fake, [0, 90, 270]), "север впереди → 0°, север слева → 90° (восток), север справа → 270° (запад)")
check(err_A.std() < 6, f"после вычитания железа разброс относительно GPS {err_A.std():.1f}° — компас заработал")
check(-20 < err_A.mean() < 0, f"среднее расхождение {err_A.mean():.1f}°: постоянный сдвиг, и он отрицательный — тема второго вопроса")

**Шаг 5.4 (0.25): переносим калибровку на другой сегмент**

Мы вычли из магнитометра железо машины — центр окружности $b$. Но откуда мы знаем, что это именно железо машины, а не какая-то особенность сегмента A? Проверить просто. Железо едет вместе с машиной, а машина между сегментами не менялась. Значит, тот же самый центр должен работать и на сегменте C — без всякого пересчёта.

Сделайте две вещи и сравните:

- примените центр из A к сегменту C как есть: `compare_mag(trip["C"], R_A, (cx, cy), ...)`;
- подгоните окружность заново по точкам самого C и примените её: `fit_circle` на магнитометре C, потом снова `compare_mag`.

In [ ]:
# YOUR CODE HERE
# (❍ᴥ❍ʋ)
# err_C = ...        # центр из A
# err_C_own = ...    # центр, подогнанный на C

In [ ]:
check(err_C.std() < 20, f"центр из A на сегменте C: разброс {err_C.std():.1f}° — хуже, чем на A, но компас всё ещё компас")
check(err_C_own.std() > 2 * err_C.std(), f"центр, подогнанный на самом C: разброс {err_C_own.std():.1f}° — окружность по короткой дуге не восстанавливается")

**Вопросы (0.5):**

1. Центр, найденный на A, работает на C, а центр, найденный на самом C, — нет, хотя он «ближе» к данным. Почему?
2. Что такое магнитное склонение и почему оно объясняет знак среднего расхождения? Какая часть расхождения им не объясняется и откуда она может браться?

**Ответ:**

__(2.5 балла) Задание 6: собираем всё вместе__

Цель этого задания — нарисовать, где ехала машина, не пользуясь GPS. Только телефон и CAN-шина. А потом положить рядом настоящий GPS-трек и посмотреть, насколько мы промахнулись.

Для этого нужны две вещи: курс машины в каждый момент времени и её скорость. Из них складывается путь — как у моряков до спутников.

- **Курс.** У нас есть два датчика, и у каждого своя дыра. Гироскоп хорошо знает, *как* курс меняется, но начинает с нуля — стартовый курс он не знает. Магнитометр знает, где север, но каждое его показание шумное. Соединим их: форму курса берём у гироскопа, а недостающую константу — у магнитометра. Разность $\psi_{\text{mag}} - \psi_{\text{gyro}}$ в каждый момент времени должна быть одним и тем же числом плюс шум; усредним её по сегменту — шум сгладится, а число и есть стартовый курс. Плюс магнитное склонение, чтобы перейти от магнитного севера к настоящему:

$$
\psi(t) = \psi_{\text{gyro}}(t) + \underbrace{\overline{\psi_{\text{mag}} - \psi_{\text{gyro}}}}_{\text{стартовый курс}} + \text{склонение}.
$$

- **Скорость** возьмём двумя способами и сравним. Первый — честный: с CAN-шины, с колёс. Второй — только по телефону: из ускорения. Для этого ускорение надо перевести в мировые оси, север и восток. У нас две матрицы перехода: $R_A$ переводит из телефона в машину, $R_z(\psi)$ из части 2 — из машины в мир, это поворот вокруг вертикали на курс. Применяем по очереди:

$$
a_{\text{мир}}(t) = R_z\big(\psi(t)\big)\, R_A\, \big(a_{\text{телефон}}(t) - \bar{a}\big).
$$

Обратите внимание, что порядок важен, $R_A$ стоит ближе к вектору и применяется первым. $R_A$ одна на всю поездку, а $R_z(\psi(t))$ своя в каждый момент — машина же поворачивает.

**Шаг 6.1 (0.25): Матрица поворота**

Напишите `rot_z(psi_deg)` — матрицу $R_z(\psi)$ для угла в градусах:

$$
R_z(\psi) = \begin{pmatrix} \cos\psi & -\sin\psi & 0 \\ \sin\psi & \cos\psi & 0 \\ 0 & 0 & 1 \end{pmatrix}.
$$

Нам понадобится своя матрица для каждого момента времени, поэтому функция должна принимать массив углов и возвращать стопку матриц формы `(n, 3, 3)`.

In [ ]:
def rot_z(psi_deg):
    p = np.radians(np.atleast_1d(psi_deg))     # углы в радианах, всегда массив
    c, s = np.cos(p), np.sin(p)
    Rz = np.zeros((len(p), 3, 3))

    # YOUR CODE HERE
    # ʕ•ᴥ•ʔ

    ...
    return Rz

In [ ]:
Rz90 = np.squeeze(rot_z(90))
check(np.allclose(Rz90 @ np.array([1, 0, 0]), [0, 1, 0]), "курс 90°: «вперёд» превращается в «восток»")
check(np.allclose(np.squeeze(rot_z(0)), np.eye(3)), "курс 0°: единичная матрица")
check(abs(np.linalg.det(Rz90) - 1) < 1e-12 and np.allclose(Rz90 @ Rz90.T, np.eye(3)), "это поворот: det = 1, R Rᵀ = I")
check(rot_z(np.array([0, 45, 90])).shape == (3, 3, 3), "массив углов → стопка матриц (n, 3, 3)")

**Шаг 6.2 (0.5): ищем курс**

Допишите в функцию недостающие строки для поиска курса:

- пересчитайте курс гироскопа на моменты времени магнитометра (`np.interp`);
- посчитайте в каждый момент времени разность между получившимися углами `angle_diff(psi_mag, psi_gyro)`;
- вычислите итоговый курс по формуле выше.

In [ ]:
DECLINATION = ...   # магнитное склонение для Сан-Франциско в 2018 году, в градусах

def fused_heading(seg, R, center, declination):
    t, psi_gyro = gyro_heading(seg, R)
    t_m, psi_mag = mag_heading(seg, R, center)

    # YOUR CODE HERE
    # (⌐■_■)

    psi_gyro_on_mag = ...    # курс гироскопа на моментах времени магнитометра
    offset = ...             # среднее angle_diff(psi_mag, psi_gyro_on_mag)
    psi = ...                # курс гироскопа + offset + склонение
    return t, psi

# готово: сравнение с GPS в движении
t_psi, psi = fused_heading(trip["A"], R_A, (cx, cy), DECLINATION)
t_gps, gps = trip["A"]["gps"]; t_v, v_can = trip["A"]["can_speed"]
moving = np.interp(t_gps, t_v, v_can) > 3
err_fused = angle_diff(np.interp(t_gps, t_psi, psi), gps[:, 4])[moving]
print(f"курс гироскоп + магнитометр vs GPS: среднее {err_fused.mean():+.1f}°, RMSE {np.sqrt((err_fused ** 2).mean()):.1f}°")

In [ ]:
check(psi.shape == t_psi.shape, "курс на сетке гироскопа")
check(abs(err_fused.mean()) < 10, f"среднее расхождение с GPS {err_fused.mean():+.1f}° — привязка от магнитометра сработала (остаток — его собственная ошибка)")
check(np.sqrt((err_fused ** 2).mean()) < 10, f"RMSE {np.sqrt((err_fused ** 2).mean()):.1f}° — форма курса от гироскопа, привязка от магнитометра, GPS не участвовал")

**Шаг 6.3: переводим GPS в метры**

В конце мы нарисуем три трека на одной картинке: два наших и GPS. Наши треки из шагов 6.4 и 6.5 будут измеряться в метрах, а GPS выдаёт градусы широты и долготы. Чтобы положить их на одну картинку и посчитать расстояние между концами треков, градусы надо перевести в метры.

Это не часть алгоритма, а подготовка эталона, поэтому функция уже написана за вас. Один градус широты — это $111\,320$ м. Один градус долготы — столько же на экваторе, но ближе к полюсам меридианы сходятся, и градус долготы ужимается в $\cos(\text{широта})$ раз.

In [ ]:
def gps_track(seg):
    t_gps, gps = seg["gps"]
    lat, lon = gps[:, 0], gps[:, 1]  # градусы
    north = (lat - lat[0]) * 111_320
    east = (lon - lon[0]) * 111_320 * np.cos(np.radians(lat[0]))
    return t_gps, np.column_stack([north, east])

t_g, pos_gps = gps_track(trip["A"])

**Шаг 6.4 (0.25): Трек по CAN-скорости**

Когда не было GPS, корабли ходили по принципу «знаешь скорость и курс — знаешь, где ты». Иногда моряки попадали таким способом в Америку вместо Индии, но да ладно.

- Курс $\psi$ мы знаем с шага 6.2;
- Скорость $v$ мы знаем с CAN-шины.

За маленький шаг времени $dt$ машина проезжает $v\,dt$ метров вдоль курса. Курс — это угол от севера по часовой стрелке, так что «вдоль курса» надо разложить на «сколько на север» и «сколько на восток». Нарисуйте: отрезок длины $v\,dt$ под углом $\psi$ к оси «север». Это гипотенуза прямоугольного треугольника, его катеты — как раз смещения:

- на север: $v\,dt \cdot \cos\psi$ — прилежащий катет;
- на восток: $v\,dt \cdot \sin\psi$ — противолежащий.

Например:
- курс $0°$ (на север) — $\cos 0 = 1$, $\sin 0 = 0$, едем только на север;
- курс $90°$ (на восток) — наоборот;
-  курс $180°$ — $\cos 180° = -1$, едем на юг, то есть на север со знаком минус.

Положение в момент $t$ — это сумма всех смещений до $t$, то есть `np.cumsum`.

В `odometry_track` скорость, шаг времени и курс в радианах уже подготовлены. Допишите три строки: смещение на север за каждый шаг, смещение на восток, накопленная сумма по обоим столбцам.

In [ ]:
def odometry_track(seg, t_psi, psi):
    t_v, v_can = seg["can_speed"]
    v = np.interp(t_psi, t_v, v_can)         # скорость на сетке курса
    dt = np.gradient(t_psi)                  # шаг по времени
    p = np.radians(psi)

    # YOUR CODE HERE
    # (づ｡◕‿‿◕｡)づ

    step_north = ...     # смещение на север за каждый шаг
    step_east = ...      # смещение на восток за каждый шаг
    pos = ...            # накопленная сумма смещений, (n, 2)
    return t_psi, pos

_, pos_odo = odometry_track(trip["A"], t_psi, psi)

In [ ]:
d_odo = np.linalg.norm(pos_odo[-1] - pos_gps[-1])
check(pos_odo.shape == (len(t_psi), 2), "трек формата (n, 2)")
check(d_odo < 40, f"конец трека по CAN-скорости в {d_odo:.0f} м от конца GPS-трека при ~190 м пути — курс без GPS работает")

**Шаг 6.5 (0.5): Трек по акселерометру**

А теперь то, ради чего всё затевалось: положение по телефону. Без GPS совсем, а от CAN-шины возьмём только одно число — скорость в самый первый момент.

Сначала про оси. Складывать ускорение имеет смысл только в мировых осях — на север и на восток. Это уже сделано в `dead_reckoning`. Ускорение телефона очищено от гравитации, повёрнуто двумя матрицами ($R_A$, потом $R_z(\psi)$) и сглажено. Получился массив `a_ne` формы `(n, 2)`: ускорение на север и на восток в каждый момент времени.

Теперь про суммирование. В шаге 6.4 у нас была скорость, и мы складывали маленькие смещения $v\,dt$ — метры. Сейчас скорости нет, есть ускорение. Но ускорение — это на сколько меняется скорость за секунду, так что $a\,dt$ — это маленькое изменение скорости, в метрах в секунду. Складываем такие изменения — получаем скорость. А из скорости, как в 6.4, — положение. Два раза `np.cumsum` подряд:

$$
\text{ускорение} \xrightarrow{\ \text{cumsum}\ } \text{скорость} \xrightarrow{\ \text{cumsum}\ } \text{положение}.
$$

Одна тонкость. В 6.4 мы начинали с нулевого положения — старт есть старт. А со скоростью так нельзя: машина в начале сегмента уже едет, и если начать с нуля, все дальнейшие скорости окажутся занижены на одну и ту же величину. Поэтому первую скорость подсмотрим у CAN-шины при $t = 0$ и направим по курсу $\psi(0)$ — через косинус и синус, как в 6.4. Дальше CAN не нужен.

Допишите три строки:

- `v0` — начальная скорость: `[v * cos(p0), v * sin(p0)]`, где `v` — скорость с CAN-шины при $t = 0$;
- `vel` — `v0` плюс накопленная сумма `a_ne * dt`;
- `pos` — накопленная сумма `vel * dt`.

Картинка с тремя треками нарисуется сама.

In [ ]:
def dead_reckoning(seg, R, t_psi, psi):
    t, acc = seg["acc"]
    t_v, v_can = seg["can_speed"]
    dt = np.gradient(t)

    # ускорение в мировых осях (север, восток, вниз)
    a_lin = acc - acc.mean(axis=0)
    M = rot_z(np.interp(t, t_psi, psi)) @ R                  # (n, 3, 3): сначала R, потом Rz
    a_world = np.einsum("nij,nj->ni", M, a_lin)              # каждую строку — своей матрицей
    a_ne = smooth(a_world[:, :2], 20)                        # север и восток, сглажено

    # YOUR CODE HERE
    # (ﾉ◕ヮ◕)ﾉ*:･ﾟ✧

    p0 = np.radians(np.interp(0, t_psi, psi))                # курс в нулевой момент
    v0 = ...          # начальная скорость: [v*cos, v*sin] при t = 0
    vel = ...         # v0 + накопленная сумма a_ne * dt
    pos = ...         # накопленная сумма vel * dt
    return t, pos

_, pos_acc = dead_reckoning(trip["A"], R_A, t_psi, psi)

# три трека на одной картинке
plt.figure(figsize=(6, 6))
plt.plot(pos_gps[:, 1], pos_gps[:, 0], "k.-", ms=3, lw=0.8, label="GPS")
plt.plot(pos_acc[:, 1], pos_acc[:, 0], lw=1.2, label="акселерометр + курс")
plt.plot(pos_odo[:, 1], pos_odo[:, 0], lw=1.5, label="CAN-скорость + курс")
plt.axis("equal"); plt.xlabel("восток, м"); plt.ylabel("север, м"); plt.legend(fontsize=9); plt.title("Сегмент A: трек без GPS"); plt.show()
print(f"акселерометр + курс: {np.linalg.norm(pos_acc[-1] - pos_gps[-1]):.0f} м от конца GPS-трека")
print(f"CAN-скорость + курс: {np.linalg.norm(pos_odo[-1] - pos_gps[-1]):.0f} м от конца GPS-трека")

In [ ]:
d_acc = np.linalg.norm(pos_acc[-1] - pos_gps[-1])
check(pos_acc.shape == (len(trip["A"]["acc"][0]), 2), "трек формата (n, 2)")
check(np.linalg.norm(pos_acc[100] - pos_gps[np.searchsorted(t_g, trip["A"]["acc"][0][100])]) < 5, "первую секунду трек по акселерометру идёт вместе с GPS")
check(d_acc > 50, f"к концу минуты трек по акселерометру уехал на {d_acc:.0f} м — это не ошибка, а физика, см. первый вопрос")

**Вопросы (1):**

1. Два трека построены с одним и тем же курсом, разница только в том, откуда взята скорость: с колёс или из суммы ускорений. Один ушёл от GPS на десяток метров, другой — на сотни. Откуда такая разница?

2. В городах всё чаще включают глушилки GPS, и навигатор на несколько минут остаётся без спутников. Приложение может продолжать вести машину по карте с помощью инерциальной навигации — это примерно тот пайплайн, который мы только что собрали. Наш вариант за минуту уехал на сотни метров куда-то вбок, на карте это соседняя улица. Пользователи будут недовольны. Что бы вы улучшили в первую очередь, чтобы траектория стала лучше и дольше держался без спутников? Назовите две идеи и для каждой скажите, какую именно ошибку она убирает.

**Ответ:**


## Раскрываем карты

Мы обещали, что в конце скажем, какой матрицей повернули данные. Она лежит ниже. Всё, что вы построили выше, держится на вашей оценке $R_A$ — пора узнать, насколько она хороша.

Если всё сделано правильно, $R_A$ и $R_{\text{true}}$ должны совпадать. Мера несовпадения двух поворотов — угол поворота матрицы $R_A R_{\text{true}}^T$, а его можно достать из следа:

$$
\cos\theta = \frac{\operatorname{tr}(R_A R_{\text{true}}^T) - 1}{2}.
$$

Посчитайте $\theta$ в градусах.

In [ ]:
R_true = np.array([[0.719884, 0.066509, 0.690901], [-0.562435, 0.639195, 0.524497], [-0.406737, -0.766164, 0.497553]])

M = R_A @ R_true.T

theta = np.degrees(np.arccos(np.clip((np.trace(M) - 1) / 2, -1, 1)))

print(f"угол между R_A и R_true: {theta:.2f}°")

## Бонус: своя поездка

__(1 балл, бонус) Задание 7\*__

Поставьте на телефон приложение [Sensor Logger](https://github.com/tszheichoi/awesome-sensor-logger) (бесплатное, iOS и Android). Положите телефон в машине, автобусе или трамвае как угодно — в карман, в подстаканник, на сиденье, — включите запись на 3–5 минут с хотя бы одним заметным поворотом и выгрузите CSV: `Accelerometer`, `Gyroscope`, `Magnetometer`, `Location`, `Orientation`.

Прогоните через свою поездку части 1 и 2. Ускорение с CAN-шины вам никто не даст, поэтому в задании 2 возьмите $\dot v$ из GPS-скорости (`Location.csv`, колонка `speed`). А вот ответ у вас будет: `Orientation.csv` — это оценка ориентации телефона, которую сделала операционная система по тем же датчикам. Сравните с ней свой курс и наклон.

**Что получилось, где разошлось с телефоном и почему:**

In [ ]:
# YOUR CODE HERE
# (❍ᴥ❍ʋ)

> **Для ревьюеров:** Бонус за осмысленность, а не за совпадение с телефоном: у GPS-скорости очень шумная производная, так что оценка `R` будет хуже, чем на CAN-шине. Полный балл — если вся цепочка прогнана, есть сравнение с `Orientation.csv` и разумное объяснение расхождений. Если запись сделана, а пайплайн не сработал и написано, почему, — 0.5.